# ⚡ Delentia OS v0.5.1: Draft Model (dspark) Training for Speculative Decoding

โน้ตบุ๊กนี้ถูกสร้างขึ้นเพื่อเป็น **ส่วนเสริม (Add-on)** สำหรับโปรเจกต์ Delentia AI SLM v0.5.1 โดยมีเป้าหมายเพื่อสร้าง **Draft Model (หรือ dspark)** ขนาดเล็ก (เช่น 1.5B หรือ 3B)

### 🎯 ทำไมต้องทำ Draft Model?
โมเดลหลักของเรามีขนาดใหญ่ถึง **27B** ซึ่งทำงานได้ช้าบนอุปกรณ์ Mobile/Edge การใช้เทคนิค **Speculative Decoding** จะช่วยแก้ปัญหานี้ โดยให้โมเดลจิ๋ว (Draft Model) ทำหน้าที่ **"เดาคำตอบล่วงหน้า"** ออกมาก่อน 4-5 คำอย่างรวดเร็ว แล้วส่งให้โมเดล 27B ตรวจทาน (Verify) รวดเดียว เทคนิคนี้จะทำให้ความเร็วในการพิมพ์ (Tokens/s) บนมือถือ **พุ่งสูงขึ้น 2-3 เท่า!**

### 🧬 กฎสำคัญของการทำ Draft Model
Draft Model จะต้องมี **"DNA ความคิด (TOON/FDIA)"** ที่เหมือนกับโมเดลหลัก 27B เป๊ะๆ เพื่อให้มันเดาใจ 27B ได้ถูก ดังนั้นเราจึงต้องนำโมเดล 1.5B มา **Fine-tune ด้วย Dataset ชุดเดียวกัน** กับที่ใช้สอน 27B ครับ

---
### 🚀 โครงสร้างการทำงาน
1. **Load Base Model:** `Qwen/Qwen2.5-1.5B` (ทำงานเร็วและฉลาดพอที่จะเดาทาง 27B ได้)
2. **Train with Delentia Dataset:** สอนมันด้วยข้อมูล Executor, Guardian, Router, และ Scribe
3. **Merge & Quantize:** แปลงเป็นไฟล์ `dspark-bf16.gguf` และ `dspark-Q4_K_M.gguf`
4. **Deployment:** นำไปโหลดคู่กับ 27B ใน llama.cpp

In [ ]:
# ==========================================
# 📦 Step 1: Install Dependencies
# ==========================================
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
import os

print('✅ Environment Ready!')

In [ ]:
# ==========================================
# 🧠 Step 2: Load Small Base Model (Qwen2.5-1.5B)
# ==========================================
max_seq_length = 4096
dtype = None
load_in_4bit = True # ใช้ 4-bit เพื่อประหยัด VRAM ระหว่างเทรน

# เราเลือก 1.5B เป็น Draft Model เพราะเร็วและเล็กพอที่จะรันคู่กับ 27B บนมือถือได้
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = os.environ.get("HF_TOKEN", "")
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = True,
)
print('✅ Model & LoRA Ready!')

In [ ]:
# ==========================================
# 📚 Step 3: Load Delentia v0.5.1 Datasets
# ==========================================
# โคลน Repo เพื่อดึง Dataset
if not os.path.exists('/content/Delentia-AI-SLM'):
    !git clone https://github.com/delentia-labs/Delentia-AI-SLM.git /content/Delentia-AI-SLM

import json
from datasets import Dataset

# เราจะนำ Dataset ทั้ง 4 ตัว (Executor, Guardian, Router, Scribe) มารวมกันเพื่อสอน Draft Model ตัวเดียว
files = [
    '/content/Delentia-AI-SLM/datasets/processed/v0.5.1/jitna_executor_pairs_v051.jsonl',
    '/content/Delentia-AI-SLM/datasets/processed/v0.5.1/jitna_guardian_pairs_v051.jsonl',
    '/content/Delentia-AI-SLM/datasets/processed/v0.5.1/jitna_router_pairs_v051.jsonl',
    '/content/Delentia-AI-SLM/datasets/processed/v0.5.1/jitna_scribe_pairs_v051.jsonl'
]

all_data = []
for f in files:
    if os.path.exists(f):
        with open(f, 'r', encoding='utf-8') as file:
            for line in file:
                if line.strip():
                    all_data.append(json.loads(line))

print(f"✅ Loaded {len(all_data)} total examples for Draft Model.")

# แปลงเป็น Dataset
def format_chat(example):
    # สมมติว่าโครงสร้าง Dataset เป็น ChatML
    if 'messages' in example:
        return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}
    # หรือถ้าเป็น prompt/completion
    elif 'prompt' in example and 'completion' in example:
        return {"text": f"<|im_start|>user\n{example['prompt']}<|im_end|>\n<|im_start|>assistant\n{example['completion']}<|im_end|>"}
    return example

dataset = Dataset.from_list(all_data)
dataset = dataset.map(format_chat)
print('✅ Dataset Formatted!')

In [ ]:
# ==========================================
# 🎓 Step 4: Fine-tune the Draft Model
# ==========================================
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # ตั้ง True ถ้าต้องการเร่งความเร็ว
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        max_steps = 150, # ปรับเพิ่มได้ตามต้องการ (แนะนำ 1 Epoch)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs_dspark",
    ),
)

print('🚀 Starting Draft Model Training...')
trainer_stats = trainer.train()
print('✅ Training Complete!')

In [ ]:
# ==========================================
# 💾 Step 5: Merge and Save to GGUF
# ==========================================
# บันทึกเป็น GGUF เลยโดยใช้ฟีเจอร์ของ Unsloth
print('⚙️ Merging and Converting to GGUF...')
model.save_pretrained_gguf("Delentia_v051_dspark_1.5B", tokenizer, quantization_method = "f16")
model.save_pretrained_gguf("Delentia_v051_dspark_1.5B", tokenizer, quantization_method = "q4_k_m")

print('✅ Draft Model GGUF Created Successfully!')
print('คุณจะได้ไฟล์: Delentia_v051_dspark_1.5B-unsloth.F16.gguf และ Delentia_v051_dspark_1.5B-unsloth.Q4_K_M.gguf')

## 🎉 วิธีใช้งาน Draft Model (dspark) ร่วมกับ 27B

หลังจากที่คุณดาวน์โหลดไฟล์ GGUF ของ 27B และไฟล์ Draft 1.5B มาแล้ว
เวลาจะรันผ่าน llama.cpp บนคอมพิวเตอร์หรือมือถือ ให้รันคำสั่งลักษณะนี้:

```bash
./llama-cli \
    -m Delentia_v051_Q1_0_G128.gguf \  <-- โมเดลหลัก 27B
    -md Delentia_v051_dspark_1.5B-unsloth.Q4_K_M.gguf \ <-- โมเดลจิ๋ว (Draft)
    -p "<|im_start|>user\nสวัสดีครับ<|im_end|>\n<|im_start|>assistant\n"
```

คุณจะเห็นเลยว่าความเร็วในการพิมพ์คำตอบ (Tokens/second) จะเพิ่มขึ้นอย่างก้าวกระโดด! 🚀